# IOAI — 2025 Lab Lab6 Defanging Lion (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!pip -q install kornia
print('사자 이미지·somenetwork.py·인코더/디코더 가중치는 노트북 셀이 공개 GCS 에서 자동 다운로드합니다(무거움).')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Defanging a lion

In [ ]:
# 추가 의존성(SSIM 손실 kornia)
!pip install -q kornia


## Problem statement

In your safari trip, you came across a dangerous lion.

In [ ]:
!curl https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/francesco-ZxNKxnR32Ng-unsplash.jpg -o lion.jpg

In [ ]:
from PIL import Image

In [ ]:
im = Image.open("lion.jpg").convert("RGB")
im

Ensure your safety by transforming the lion in this picture into a slightly less dangerous animal: an african buffalo.

Here is how your work will be scored:

- 1 pt awarded for demonstrating that your transformed lion is closer in meaning to the text "a peaceful african buffalo" than the text "a prowling lion on the hunt" by `openai/clip-vit-base-patch32`

> For CLIP scoring, use cosine similarity for your calculation. Scoring less than 0.5 using the reference implementation of binary softmax is sufficient to be considered as closer to buffalo. To ensure that your implementation is correct, and to save you the headache of losing progress due to not realizing you need to normalize your vectors, sample code is provided below as a reference. I am not mandating you use this code because you might prefer to handle the vectors directly instead of being wrapped in transformers as a dictionary with key "pixel_values"

- 1 pt awarded if the point above is earned, and your transformed lion has less than 0.3 SSIM loss compared to the original lion.

> For SSIM loss, install `kornia` and look at the sample code provided below.

- 1 pt awarded if `openai/clip-vit-base-patch32` considers the softmaxed probability that your transformed lion is a buffalo is higher than 0.7 (i.e. lion < 0.3) AND SSIM loss is less than 0.3
- 2 pts awarded for demonstrating that your transformed lion is so convincingly transformed into a buffalo, that it visually looks more like a buffalo than a lion. SSIM loss criterion not applicable for these points.
    - If it clearly looks like a buffalo to me, these pts will count. If not, I will load any readily-available image classification model pretrained on ImageNet and check if the transformed image is closer to class 291 (lion) or class 346 (water buffalo). The reason for this evaluation is to make sure that you aren't creating an adversarial image that is barely modified but scores the correct class.
    - Make sure that your image is shown in your notebook for further inspection.
- 1 pt awarded for explaining your thought process and reasoning for your work done. Keep it brief, one short paragraph is enough!

Partial credit will be granted at discretion.

You are allowed to use:

- any pretrained model from `torchvision` and `transformers`
- any code and pretrained models made available to you as part of this problem
- standard ML libraries (e.g. numpy, sklearn, albumentations, einops)
- additional images of your own

You are prohibited to use:

- pretrained models that are not from the allowed sources. Off the shelf models, dedicated libraries and github repos are not allowed.
- responses from LLM APIs like OpenAI and using the response as your solution
- hosted local LLMs to generate images and use the generated image as your solution
- any technique that amounts to replacing the lion with a different picture

You might find this mysterious SomeNetwork useful.

Requirements: 
- Install `attrs`. You should already have `numpy`, `torch`, and `torchvision`
- Download `somenetwork.py` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/somenetwork.py
- Download weights (370+ MB combined):
    - `encoder.pt` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/encoder.pt
    - `decoder.pt` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/decoder.pt
- Download `using-somenetwork.ipynb` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/using-somenetwork.ipynb to see how to use it

In [ ]:
# somenetwork + 인코더/디코더 가중치 자동 다운로드(공개 GCS, 총 ~370MB)
!pip install -q attrs
import os, urllib.request
B = "https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion"
for _f in ["somenetwork.py", "encoder.pt", "decoder.pt"]:
    if not os.path.exists(_f):
# 참조용 버팔로 이미지(GCS 엔 없어 jaredliw 저장소에서)
if not os.path.exists("my_buffalo.jpeg"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/jaredliw/ioai-tsp-2025/main/lab6/defanging-lion/my_buffalo.jpeg", "my_buffalo.jpeg")
        print("downloading", _f, "...")
        urllib.request.urlretrieve(f"{B}/{_f}", _f)
print("준비 완료:", [f for f in ["somenetwork.py","encoder.pt","decoder.pt","lion.jpg"] if os.path.exists(f)])


## Sample code to help you get started

```python
device = "cuda" if torch.cuda.is_available() else "cpu"

from PIL import Image
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

text_inputs = processor(
    text="a peaceful african buffalo", return_tensors="pt", padding=True
).to(device)
buffalo_text_embeddings = model.get_text_features(**text_inputs).float()
buffalo_text_embeddings = F.normalize(buffalo_text_embeddings, p=2, dim=-1)

text_inputs = processor(text="a prowling lion on the hunt", return_tensors="pt", padding=True).to(
    device
)
lion_text_embeddings = model.get_text_features(**text_inputs).float()
lion_text_embeddings = F.normalize(lion_text_embeddings, p=2, dim=-1)

image = Image.open("lion.jpg").convert("RGB")
inputs = processor(
    images=image,
    return_tensors="pt",
    padding=True,
    do_rescale=True,
    do_normalize=True,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

def compute_clip_lion2buffalo_loss(image_inputs):
    image_embeddings = model.get_image_features(**image_inputs).float()
    image_embeddings = F.normalize(image_embeddings, p=2, dim=-1)

    global lion_text_embeddings
    global buffalo_text_embeddings

    # Cosine similarity (higher = more similar)
    lion_sim = (image_embeddings @ lion_text_embeddings.T).squeeze()
    buffalo_sim = (image_embeddings @ buffalo_text_embeddings.T).squeeze()

    scores = torch.stack([lion_sim, buffalo_sim])
    probs = torch.softmax(scores, dim=0)
    lion_score = probs[0]
    
    # <0.5 means closer to buffalo
    return lion_score
```

```python
from kornia.losses import SSIMLoss

ssim_loss_fxn = SSIMLoss(window_size=7).to(device)

a = torch.rand(1, 3, 256, 256)
b = torch.rand(1, 3, 256, 256)

ssim_loss_fxn(a, b)
```

## Your work below

In [ ]:
# Read everything clearly before you start!
import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from kornia.losses import SSIMLoss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

In [ ]:
def to_text_embedding(text):
    clip_inputs = processor(
        text=text, 
        return_tensors="pt", 
        padding=True
    ).to(device)
    _o = model.get_text_features(**clip_inputs)
    text_embs = (_o.pooler_output if hasattr(_o, "pooler_output") else _o).float()  # transformers 5.x 대응
    text_embs = F.normalize(text_embs, p=2, dim=-1)
    return text_embs

In [ ]:
buffalo_text = "a peaceful african buffalo"
buffalo_emb = to_text_embedding(buffalo_text).detach()

lion_text = "a prowling lion on the hunt"
lion_emb = to_text_embedding(lion_text).detach()

In [ ]:
def clip_preprocess(image, do_rescale=True):
    inputs = processor(
        images=image,
        return_tensors="pt",
        padding=True,
        do_rescale=do_rescale,
        do_normalize=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    return inputs

In [ ]:
def compute_clip_lion2buffalo_loss(clip_inputs):
    _o = model.get_image_features(**clip_inputs)
    embs = (_o.pooler_output if hasattr(_o, "pooler_output") else _o).float()  # transformers 5.x 대응
    embs = F.normalize(embs, p=2, dim=-1)

    global buffalo_emb, lion_emb
    # Cosine similarity (higher = more similar)
    lion_sim = (embs @ lion_emb.T).squeeze()
    buffalo_sim = (embs @ buffalo_emb.T).squeeze()

    scores = torch.stack([lion_sim, buffalo_sim])
    probs = torch.softmax(scores, dim=0)
    lion_score = probs[0]
    
    # <0.5 means closer to buffalo
    return lion_score

In [ ]:
def compute_ssim_loss(tensor1, tensor2):  # Each tensor is an image with shape [C, H, W]
    ssim_loss_fxn = SSIMLoss(window_size=7).to(device)
    return ssim_loss_fxn(tensor1.unsqueeze(0), tensor2.unsqueeze(0))

---

In [ ]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from somenetwork import Encoder, Decoder, map_pixels, unmap_pixels

In [ ]:
enc = Encoder().to(device)
with open("encoder.pt", "rb") as f:
    enc.load_state_dict(torch.load(f, map_location=device))

dec = Decoder().to(device)
with open("decoder.pt", "rb") as f:
    dec.load_state_dict(torch.load(f, map_location=device))

In [ ]:
def resize_and_crop(image, target_image_size=224):
    s = min(image.size)
    
    if s < target_image_size:
        raise ValueError(f'min dim for image {s} < {target_image_size}')
        
    r = target_image_size / s
    s = (round(r * image.size[1]), round(r * image.size[0]))
    
    image = TF.resize(image, s, interpolation=Image.LANCZOS)
    image = TF.center_crop(image, output_size=2 * [target_image_size])
    return image

In [ ]:
def vae_preprocess(image):
    image = resize_and_crop(image)
    image = torch.unsqueeze(T.ToTensor()(image), 0)
    return map_pixels(image)

In [ ]:
def get_enc_latent(image):
    x = vae_preprocess(image).to(device)
    z_logits = enc(x)
    return z_logits

In [ ]:
def generate_image(z):
    x_stats = dec(z).float()
    x_rec = unmap_pixels(torch.sigmoid(x_stats[:, :3]))[0]
    return x_rec

In [ ]:
def default_transform(z_logits):  # Given in using-somenetwork.ipynb
    z = torch.argmax(z_logits, dim=1)
    z = F.one_hot(z, num_classes=enc.vocab_size).permute(0, 3, 1, 2).float()
    return z

def morph(image, transform=default_transform):
    z_logits = get_enc_latent(image)
    z = transform(z_logits)
    return generate_image(z)

In [ ]:
image2tensor = T.ToTensor()  # This is a function
tensor2image = T.ToPILImage(mode="RGB")  # This is a function

## Demo

This section makes sure everything works fine.

In [ ]:
im_orig = resize_and_crop(im)  # Will use this instead of the above
im_tensor_orig = image2tensor(im_orig).to(device)
im_orig

In [ ]:
compute_clip_lion2buffalo_loss(clip_preprocess(im_orig)).item()

In [ ]:
im_rec1 = tensor2image(morph(im_orig))
im_rec1

In [ ]:
compute_clip_lion2buffalo_loss(clip_preprocess(im_rec1)).item()

In [ ]:
compute_ssim_loss(im_tensor_orig, image2tensor(im_rec1).to(device)).item()

## Main

In [ ]:
import os
import heapq
import torch.optim as optim
from uuid import uuid4
from torchvision.models import resnet18, ResNet18_Weights
from IPython.display import display

In [ ]:
im_buff = Image.open("my_buffalo.jpeg").convert("RGB")  # Get this from the Internet
im_buff = resize_and_crop(im_buff)
im_buff_tensor = image2tensor(im_buff).to(device)
im_buff

In [ ]:
Image.blend(im_orig, im_buff, 0.6)

Align the two animals carefully to achieve the best training outcome!

In [ ]:
compute_clip_lion2buffalo_loss(clip_preprocess(im_buff)).item()

In [ ]:
z_buff = torch.argmax(get_enc_latent(im_buff), dim=1)
z_buff_one_hot = F.one_hot(z_buff, num_classes=enc.vocab_size).permute(0, 3, 1, 2).float()
z_buff.shape, z_buff_one_hot.shape

In [ ]:
resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in resnet.parameters():
    p.requires_grad_(False)
resnet = torch.nn.Sequential(*list(resnet.children())[:5]).to(device)  # up to layer2
resnet.eval();

In [ ]:
def compute_perceptual_loss(x_gen, x_target):
    return F.mse_loss(resnet(x_gen.unsqueeze(0)), resnet(x_target.unsqueeze(0)))

In [ ]:
def hard_binary(x):
    hard = (x >= 0.5).float()
    return (hard - x).detach() + x

def hard_binary_flipped(x):
    hard = (x < 0.5).float()
    return (hard - x).detach() + x

In [ ]:
class TopK:
    def __init__(self, k):
        self.k = k
        self.heap = []  # min-heap of (score, item)

    def add(self, score, item):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, (score, item))
        else:
            # only add if the new score is better than the smallest in heap
            if score > self.heap[0][0]:
                heapq.heappushpop(self.heap, (score, item))

    def get_top_k(self):
        return sorted(self.heap, key=lambda x: -x[0])  # (score, item), sorted by score desc

In [ ]:
best_40 = TopK(40)

In [ ]:
def my_transform(z_logits, num_epochs=2000):
    z = torch.argmax(z_logits, dim=1)
    z = F.one_hot(z, num_classes=enc.vocab_size).permute(0, 3, 1, 2).float()
    z.requires_grad_(False)

    keep_mask = torch.randn(z_buff.shape).unsqueeze(1).to(device)
    keep_mask.requires_grad_(True)

    optimizer = optim.Adam([keep_mask], lr=0.8)
    for epoch in range(num_epochs):
        z_edited = hard_binary(keep_mask) * z + hard_binary_flipped(keep_mask) * z_buff_one_hot
        x_rec = generate_image(z_edited)

        perceptual_loss = compute_perceptual_loss(x_rec, im_buff_tensor)
        ssim_loss = compute_ssim_loss(im_tensor_orig, x_rec)
        total_loss = perceptual_loss + ssim_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        best_40.add(-total_loss, x_rec)

        if epoch % 100 == 0 or epoch == num_epochs - 1:
            with torch.no_grad():
                num_changed = hard_binary_flipped(keep_mask).sum().int()
                print(f"Epoch {epoch + 1}: perceptual loss - {perceptual_loss.item()}, SSIM loss - {ssim_loss.item()}, total loss - {total_loss.item()}, changed - {num_changed.item()}")

    return z_edited

In [ ]:
im_rec2 = tensor2image(morph(im_orig, my_transform))
im_rec2

In [ ]:
output_folder_dir = f"./my_output_{str(uuid4()).split("-")[0]}"
os.makedirs(output_folder_dir, exist_ok=True)

In [ ]:
for idx, (_, tensor_rec_test) in enumerate(best_40.get_top_k()):
    im_rec_test = tensor2image(tensor_rec_test)

    clip_loss = compute_clip_lion2buffalo_loss(clip_preprocess(im_rec_test)).item()
    ssim_loss = compute_ssim_loss(image2tensor(im_orig).to(device), tensor_rec_test).item()
    if clip_loss < 0.5 and ssim_loss < 0.3:
        print(f"Idx {idx}: {clip_loss=}, {ssim_loss=}")
        im_rec_test.save(f"./{output_folder_dir}/{idx}.jpeg")
        display(im_rec_test)

## Explanation

We implemented a latent space mixing approach to transform a lion image into a buffalo. Through gradient-based optimization, we learned a binary mask that selectively replaces parts of the lion's latent code with corresponding regions from the buffalo's latent code. The optimization objective combines perceptual loss (using ResNet features) and SSIM loss to maintain visual quality while encouraging buffalo-like characteristics.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)